# 02 - Feature Engineering (Rossmann Store Sales)

**Goal:** turn the cleaned data into model-ready features for daily sales forecasting.

| | |
|---|---|
| **Input** | `data/processed/cleaned_data.csv` |
| **Output** | `data/processed/featured_data.csv` |
| **Reusable code** | `src/features.py` (all functions used below) |

**Rules followed in this notebook**
- No leakage: lag and rolling features use only earlier dates.
- The split is chronological, never random.
- The preprocessing pipeline is fitted on the training split only.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import issparse

sys.path.append(str(Path("..") / "src"))
import features as ft

pd.set_option("display.max_columns", 50)

DATA_DIR = Path("../data/processed")

## 2. Load and clean

Checks required columns, fixes dtypes, drops incomplete rows, sorts by store and date,
and stops if any store has two rows on the same date.

In [ ]:
df = ft.load_data(DATA_DIR / "cleaned_data.csv", store_path=Path("../data/raw/store.csv"))
df = ft.clean_data(df)

print("Rows:", len(df))
print("Stores:", df[ft.STORE].nunique())
print("Dates:", df[ft.DATE].min().date(), "->", df[ft.DATE].max().date())
df.head()

## 3. Lag features

`Sales_Lag_7`, `Sales_Lag_14`, `Sales_Lag_28` = sales of the same store exactly 7 / 14 / 28 **calendar days** earlier.
They are matched by date (not by row position), so a missing day gives `NaN` instead of a wrong value.
The first 7 / 14 / 28 days of each store have no history, so they are `NaN`; the pipeline imputes them later.

In [ ]:
df_feat = ft.add_lag_features(df)
df_feat[[ft.STORE, ft.DATE, ft.TARGET, "Sales_Lag_7", "Sales_Lag_14", "Sales_Lag_28"]].head(10)

## 4. Rolling statistics

Mean, std, min and max of the **previous 28 days** for each store.
The current day is excluded (`closed="left"`), so today's sales never leak into today's features.

In [ ]:
df_feat = ft.add_rolling_features(df_feat)
rolling_cols = [c for c in df_feat.columns if c.startswith("Sales_Rolling_")]
df_feat[[ft.STORE, ft.DATE, ft.TARGET] + rolling_cols].head(10)

## 5. Calendar features

`Day`, `Month`, `Year`, `DayOfWeek` (Monday = 0) and `IsWeekend` (Saturday/Sunday).
Holiday information comes from the existing `StateHoliday` and `SchoolHoliday` columns.
Store characteristics (`StoreType`, `Assortment`, competition and promo columns) are already in the dataset.

In [ ]:
df_feat = ft.add_calendar_features(df_feat)
df_feat[[ft.DATE, "Day", "Month", "Year", "DayOfWeek", "IsWeekend"]].head()

## 6. Choose model inputs

Excluded on purpose:
- `Sales` - the target.
- `Date` - replaced by the calendar features.
- `Customers` - not known before the day happens.
- `Open` - constant after closed days were removed.
- `IsOutlier` - derived from `Sales` (leakage).
- `Year` - test years do not appear in training.

`Store` is treated as a category, not as a number.

In [ ]:
numeric_cols, categorical_cols = ft.get_feature_columns(df_feat)

print("Numeric    :", numeric_cols)
print("Categorical:", categorical_cols)

## 7. Time-based split

Oldest 70% of dates for training, next 15% for validation, last 15% for testing.
Boundaries use unique dates, so one date never falls in two splits.

In [ ]:
train_df, valid_df, test_df = ft.time_split(df_feat)

for name, part in [("Train", train_df), ("Validation", valid_df), ("Test", test_df)]:
    print(f"{name:<11} rows: {len(part):>9,}   {part[ft.DATE].min().date()} -> {part[ft.DATE].max().date()}")

## 8. Preprocessing pipeline

`ColumnTransformer` with two branches:
- **Numeric:** median imputation, then `StandardScaler`.
- **Categorical:** most-frequent imputation, then `OneHotEncoder` (unknown categories ignored).

It is fitted on the training split only, then applied unchanged to validation and test.

In [ ]:
feature_cols = numeric_cols + categorical_cols

X_train, y_train = train_df[feature_cols], train_df[ft.TARGET]
X_valid, y_valid = valid_df[feature_cols], valid_df[ft.TARGET]
X_test, y_test = test_df[feature_cols], test_df[ft.TARGET]

preprocessor = ft.build_preprocessor(numeric_cols, categorical_cols)
X_train_t = preprocessor.fit_transform(X_train)

print("Transformed train shape:", X_train_t.shape)
values = X_train_t.data if issparse(X_train_t) else X_train_t
print("Any NaN left:", np.isnan(values).any())

## 9. Sanity check

A quick Ridge model only to confirm that the pipeline runs end to end and the features carry signal.
**These numbers are not the project results**; the real models are trained in the modelling notebooks.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

check = Pipeline([("prep", preprocessor), ("model", Ridge(alpha=1.0))])
check.fit(X_train, y_train)

rows = []
for name, X, y in [("Validation", X_valid, y_valid), ("Test", X_test, y_test)]:
    pred = check.predict(X)
    rows.append({
        "Split": name,
        "MAE": mean_absolute_error(y, pred),
        "RMSE": mean_squared_error(y, pred) ** 0.5,
        "R2": r2_score(y, pred),
    })
pd.DataFrame(rows).round(3)

## 10. Save the featured dataset

In [ ]:
OUT_PATH = DATA_DIR / "featured_data.csv"
df_feat.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print("Shape:", df_feat.shape)

## How to use this in the modelling notebooks / scripts

```python
import sys
sys.path.append("../src")
from features import (load_data, clean_data, build_features, time_split,
                      get_feature_columns, build_preprocessor, TARGET)

df = build_features(clean_data(load_data("../data/processed/cleaned_data.csv")))
train_df, valid_df, test_df = time_split(df)
numeric_cols, categorical_cols = get_feature_columns(df)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
```

**Notes for the next steps**
- **Polynomial Regression:** apply `PolynomialFeatures` to `numeric_cols` only. Applying it to the one-hot `Store` columns creates thousands of features.
- **Forecast horizon:** rolling features use sales up to the previous day, so predictions are valid one day ahead. Lags of 7+ days alone would allow a 7-day horizon.
- **Missing values:** `NaN` in the first days of each store is expected and handled by the imputers.